# CLIP Linear Probe on MEDIC — CrisisLens（HF Streaming 版）

用 **MEDIC** 災害影像，在凍結的 **CLIP ViT-L/14** 特徵上訓練線性分類器，
解決 zero-shot 分不開「地震 vs 土石流」等視覺相近類別的問題。

**關鍵**：MEDIC 的圖片直接內嵌在 HuggingFace 的 parquet（共 10.9GB）。
用 `streaming=True` **邊串流邊抽特徵**，11GB 從網路流過即丟，**完全不需下載到 Drive**，
最後只留下約 200MB 的特徵快取。

**流程**：HF 串流 → 抽 CLIP 特徵（T4 GPU，一次性）→ 訓練 linear probe → 評估 → 匯出 `clip_linear_head.pth`

⚠️ 執行前：**Runtime → Change runtime type → T4 GPU**


## 0. 環境檢查 + 安裝套件

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
assert torch.cuda.is_available(), "請先把 Runtime 改成 T4 GPU！"

!pip -q install ftfy regex tqdm datasets
!pip -q install git+https://github.com/openai/CLIP.git


## 1.（選用）掛 Drive 存「特徵快取」

只存約 200MB 的 `.npz` 特徵，不是 11GB。掛了之後重跑可直接讀快取、不用重抽。
不想掛也行，改用 `/content`（session 結束會消失）。


In [ ]:
import os
USE_DRIVE = True   # 不想掛 Drive 就改成 False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = '/content/drive/MyDrive/crisislens_medic'
else:
    CACHE_DIR = '/content/cache'
os.makedirs(CACHE_DIR, exist_ok=True)
print("特徵快取目錄:", CACHE_DIR)


## 2. 串流連到 HF，檢查資料結構

第一次跑會偵測 `disaster_types` 的類別名稱與圖片欄位，確認 schema。


In [ ]:
from datasets import load_dataset

probe = load_dataset("QCRI/MEDIC", split="train", streaming=True)
ex = next(iter(probe))
print("欄位:", list(ex.keys()))
print("image 型別:", type(ex.get("image")))

# 取得 disaster_types 的類別名稱
feat = probe.features.get("disaster_types") if probe.features else None
DT_NAMES = feat.names if (feat is not None and hasattr(feat, "names")) else None
print("disaster_types 類別:", DT_NAMES)
print("第一筆 disaster_types 值:", ex.get("disaster_types"))


## 3. 類別映射：MEDIC 7 類 → CrisisLens 6 類

In [ ]:
# 必須跟本機 utils/config.py 的 CLASSES_EN 順序一致！
CLASSES_EN = [
    "Earthquake Damage",       # 0
    "Flood",                   # 1
    "Fire",                    # 2
    "Typhoon or Storm Damage", # 3
    "Landslide",               # 4
    "Other or No Disaster",    # 5
]

# MEDIC disaster_types（小寫）-> CrisisLens index
MEDIC_MAP = {
    "earthquake":     0,
    "flood":          1,
    "fire":           2,
    "hurricane":      3,
    "landslide":      4,
    "not_disaster":   5,
    "other_disaster": 5,
}

def dt_to_idx(val):
    """把一筆 disaster_types（可能是 int 或字串）轉成 CrisisLens index，無法對應回 None。"""
    name = DT_NAMES[val] if (DT_NAMES is not None and isinstance(val, int)) else str(val)
    return MEDIC_MAP.get(name.strip().lower())

# 驗證映射覆蓋所有類別
if DT_NAMES:
    print("映射檢查:")
    for n in DT_NAMES:
        print(f"  {n:16s} -> {MEDIC_MAP.get(n.strip().lower())}")


## 4. 串流抽 CLIP 特徵（一次性，存成 .npz 快取）

每張圖只 forward 一次。11GB 從 HF 串流過去、抽完即丟，只留特徵。


In [ ]:
import clip, numpy as np, torch
from PIL import Image, ImageFile
from tqdm import tqdm
ImageFile.LOAD_TRUNCATED_IMAGES = True

device = "cuda"
model, preprocess = clip.load("ViT-L/14", device=device)   # 與本機 CLIP_MODEL_NAME 對齊
model.eval()
FEAT_DIM = model.visual.output_dim   # ViT-L/14 = 768

@torch.no_grad()
def extract_split(split, batch=64):
    cache = f"{CACHE_DIR}/feat_{split}.npz"
    if os.path.exists(cache):
        d = np.load(cache); print(f"{split}: 讀快取 {d['X'].shape}")
        return d["X"], d["y"]

    ds = load_dataset("QCRI/MEDIC", split=split, streaming=True)
    feats, labels, buf_i, buf_l = [], [], [], []
    def flush():
        if not buf_i: return
        x = torch.stack(buf_i).to(device)
        f = model.encode_image(x)
        f = f / f.norm(dim=-1, keepdim=True)        # L2 normalize（推論端要一致）
        feats.append(f.cpu().numpy().astype("float32"))
        labels.extend(buf_l); buf_i.clear(); buf_l.clear()

    for ex in tqdm(ds, desc=split):
        idx = dt_to_idx(ex.get("disaster_types"))
        if idx is None:
            continue
        img = ex["image"]
        if img.mode != "RGB":
            img = img.convert("RGB")
        buf_i.append(preprocess(img)); buf_l.append(idx)
        if len(buf_i) >= batch:
            flush()
    flush()

    X = np.concatenate(feats); y = np.array(labels)
    np.savez(cache, X=X, y=y); print(f"{split}: 抽完 {X.shape} -> {cache}")
    return X, y

Xtr, ytr = extract_split("train")
Xdv, ydv = extract_split("dev")
Xte, yte = extract_split("test")

print("\n各類別張數（train）：")
for i, name in enumerate(CLASSES_EN):
    print(f"  [{i}] {name:26s} {(ytr==i).sum()}")


## 5. 訓練 Linear Probe

特徵已快取，全量梯度下降在 GPU 上瞬間完成。用 **class weight** 緩解類別不平衡
（`not_disaster` 偏多、`landslide` 偏少）。


In [ ]:
import torch.nn as nn, numpy as np

dev_t = "cuda"
Xtr_t = torch.tensor(Xtr, device=dev_t); ytr_t = torch.tensor(ytr, device=dev_t)
Xdv_t = torch.tensor(Xdv, device=dev_t); ydv_t = torch.tensor(ydv, device=dev_t)

counts = np.bincount(ytr, minlength=len(CLASSES_EN))
cw = counts.sum() / (len(CLASSES_EN) * np.maximum(counts, 1))
class_w = torch.tensor(cw, dtype=torch.float32, device=dev_t)
print("class weights:", np.round(cw, 3))

clf   = nn.Linear(FEAT_DIM, len(CLASSES_EN)).to(dev_t)
opt   = torch.optim.AdamW(clf.parameters(), lr=1e-3, weight_decay=1e-3)
lossf = nn.CrossEntropyLoss(weight=class_w)

best_acc, best_state = 0.0, None
for epoch in range(1, 301):
    clf.train(); opt.zero_grad()
    loss = lossf(clf(Xtr_t), ytr_t); loss.backward(); opt.step()
    if epoch % 10 == 0:
        clf.eval()
        with torch.no_grad():
            acc = (clf(Xdv_t).argmax(1) == ydv_t).float().mean().item()
        if acc > best_acc:
            best_acc = acc; best_state = {k: v.clone() for k, v in clf.state_dict().items()}
        print(f"epoch {epoch:3d}  loss {loss.item():.4f}  dev_acc {acc:.4f}")

clf.load_state_dict(best_state)
print(f"\n最佳 dev acc: {best_acc:.4f}")


## 6. 在 test 集評估（重點看地震 vs 土石流混淆）

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

clf.eval()
with torch.no_grad():
    pred = clf(torch.tensor(Xte, device=dev_t)).argmax(1).cpu().numpy()

print(classification_report(yte, pred, target_names=CLASSES_EN, zero_division=0))
print("Confusion matrix (row=真實, col=預測):")
cm = confusion_matrix(yte, pred, labels=list(range(len(CLASSES_EN))))
print("      " + " ".join(f"{i:>5d}" for i in range(len(CLASSES_EN))))
for i, row in enumerate(cm):
    print(f"[{i}] " + " ".join(f"{v:>5d}" for v in row), CLASSES_EN[i])


## 7. 匯出權重 → 下載放本機 `CrisisLens/models/`

In [ ]:
ckpt = {
    "state_dict": {k: v.cpu() for k, v in clf.state_dict().items()},
    "classes_en": CLASSES_EN,
    "clip_model": "ViT-L/14",
    "in_dim":     FEAT_DIM,
}
out_path = "clip_linear_head.pth"
torch.save(ckpt, out_path)
torch.save(ckpt, f"{CACHE_DIR}/clip_linear_head.pth")   # 備份
print("已存:", out_path)

from google.colab import files
files.download(out_path)


## 8. 本機整合（已幫你寫好）

1. 把下載的 `clip_linear_head.pth` 放到 `CrisisLens/models/`
2. 重啟 streamlit，首頁左側「CLIP Prompt Set」會自動多出 **「E｜Linear Probe（MEDIC訓練）」**（首選）
3. `clip_classifier.py` 的 `classify_linear_probe()` 會自動載入並推論

> 推論端用同一個 `ViT-L/14` + 同樣的 L2 normalize，確保與訓練一致。
